In [1]:
# Install the libraries required for BERT, dataset handling, and evaluation.

!pip -q install -U transformers datasets accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.8 MB/s eta 0:00:00


In [3]:
# Install and import the required libraries for BERT sentiment analysis.

import random
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

Device: cuda


In [5]:
# Load the IMDB dataset, select a sample, and check the class distribution.

dataset = load_dataset("stanfordnlp/imdb")

TRAIN_SIZE = 2000
TEST_SIZE = 500

train_ds = dataset["train"].shuffle(
    seed=SEED
).select(range(TRAIN_SIZE))

test_ds = dataset["test"].shuffle(
    seed=SEED
).select(range(TEST_SIZE))

print("Training samples:", len(train_ds))
print("Testing samples:", len(test_ds))

print("\nClass Distribution:")
print(pd.Series(train_ds["label"]).value_counts().sort_index())

print("\nExample Review:")
print(train_ds[0]["text"][:300])
print("Label:", train_ds[0]["label"])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training samples: 2000
Testing samples: 500

Class Distribution:
0    1000
1    1000
Name: count, dtype: int64

Example Review:
There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier's plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot 
Label: 1


In [6]:
# Tokenize the reviews and load the pretrained BERT model for binary classification.

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_ds.map(tokenize_function, batched=True)
tokenized_test = test_ds.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

def create_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

print("Tokenizer and model configuration ready.")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizer and model configuration ready.


In [7]:
# Define evaluation metrics and a reusable function for training BERT.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


def train_model(epochs, output_dir):
    global model

    model = create_model()

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate()

    return trainer, metrics

In [8]:
# Train BERT for one and two epochs, then compare the evaluation results.

trainer_1, metrics_1 = train_model(
    epochs=1,
    output_dir="./bert_1_epoch"
)

trainer_2, metrics_2 = train_model(
    epochs=2,
    output_dir="./bert_2_epochs"
)

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score"],

    "1 Epoch": [
        metrics_1["eval_accuracy"],
        metrics_1["eval_precision"],
        metrics_1["eval_recall"],
        metrics_1["eval_f1"]
    ],

    "2 Epochs": [
        metrics_2["eval_accuracy"],
        metrics_2["eval_precision"],
        metrics_2["eval_recall"],
        metrics_2["eval_f1"]
    ]
})

print(comparison)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.296404,0.291218,0.872000,0.847328,0.902439,0.874016


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.205970,0.376323,2,0.888000,0.892562,0.878049,0.885246


      Metric   1 Epoch  2 Epochs
0   Accuracy  0.872000  0.888000
1  Precision  0.847328  0.892562
2     Recall  0.902439  0.878049
3   F1-Score  0.874016  0.885246


In [9]:
# Generate a classification report and test BERT on custom movie reviews.

final_trainer = trainer_2

pred_output = final_trainer.predict(tokenized_test)
predictions = np.argmax(pred_output.predictions, axis=-1)

print(classification_report(
    pred_output.label_ids,
    predictions,
    target_names=["NEGATIVE", "POSITIVE"],
    digits=4
))


def predict_sentiment(texts):
    final_trainer.model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

    inputs = {
        k: v.to(final_trainer.model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = final_trainer.model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

    for text, pred, probs in zip(texts, predictions, probabilities):
        print(f"\nText: {text}")
        print(f"Sentiment: {id2label[pred.item()]}")
        print(f"Confidence: {probs[pred].item():.2%}")


sample_texts = [
    "The movie was amazing and the acting was excellent.",
    "The story was boring and the movie was disappointing.",
    "The visuals were great but the plot was weak."
]

predict_sentiment(sample_texts)

              precision    recall  f1-score   support

    NEGATIVE     0.8837    0.8976    0.8906       254
    POSITIVE     0.8926    0.8780    0.8852       246

    accuracy                         0.8880       500
   macro avg     0.8881    0.8878    0.8879       500
weighted avg     0.8881    0.8880    0.8880       500


Text: The movie was amazing and the acting was excellent.
Sentiment: POSITIVE
Confidence: 98.59%

Text: The story was boring and the movie was disappointing.
Sentiment: NEGATIVE
Confidence: 98.64%

Text: The visuals were great but the plot was weak.
Sentiment: NEGATIVE
Confidence: 98.22%


In [10]:
# Save the selected fine-tuned BERT model and tokenizer for future use.

SAVE_PATH = "./bert_sentiment_model"

final_trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: ./bert_sentiment_model
